# `vasprun.xml` snapshot to LAMMPS

Select one snapshot from a VASP MD `vasprun.xml`, estimate its velocities by finite differences,
and export it as a LAMMPS data file.

The notebook now looks for candidate files in `input/vasprun/` by default.


In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
if (cwd / "python" / "lammps_md_preparation" / "__init__.py").exists():
    NOTEBOOK_DIR = cwd
    package_root = cwd / "python"
elif (cwd / "lammps_md_preparation" / "python" / "lammps_md_preparation" / "__init__.py").exists():
    NOTEBOOK_DIR = cwd / "lammps_md_preparation"
    package_root = NOTEBOOK_DIR / "python"
else:
    raise RuntimeError("Run this notebook from the repository root or from lammps_md_preparation/.")

if str(package_root) not in sys.path:
    sys.path.insert(0, str(package_root))

from lammps_md_preparation.utils import extract_vasprun_snapshot_to_lammps, specorder_from_atoms


In [ ]:
VASPRUN_DIR = NOTEBOOK_DIR / "input" / "vasprun"
VASPRUN_PATTERN = "*.xml"
VASPRUN_INDEX = 0
SNAPSHOT_INDEX = -1
ION_TIMESTEP_FS = 1.0

OUTPUT_DIR = NOTEBOOK_DIR / "output" / "vasprun_to_lammps"
OUTPUT_FILENAME = "lmp_snapshot.lmp"
ATOM_STYLE = "atomic"
UNITS = "metal"
OVERRIDE_SPECORDER = None


In [ ]:
vasprun_files = sorted(VASPRUN_DIR.glob(VASPRUN_PATTERN))
if not vasprun_files:
    raise FileNotFoundError(
        f"No vasprun file matched '{VASPRUN_PATTERN}' in {VASPRUN_DIR}"
    )
if VASPRUN_INDEX >= len(vasprun_files):
    raise IndexError(
        f"VASPRUN_INDEX={VASPRUN_INDEX} but only {len(vasprun_files)} matching file(s) were found."
    )

for idx, candidate in enumerate(vasprun_files):
    print(f"[{idx}] {candidate}")

VASPRUN_FILE = vasprun_files[VASPRUN_INDEX]
print(f"Selected vasprun file: {VASPRUN_FILE}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_path = OUTPUT_DIR / OUTPUT_FILENAME

atoms = extract_vasprun_snapshot_to_lammps(
    vasprun_file=VASPRUN_FILE,
    snapshot_index=SNAPSHOT_INDEX,
    ionic_timestep_fs=ION_TIMESTEP_FS,
    output_path=output_path,
    specorder=OVERRIDE_SPECORDER,
    atom_style=ATOM_STYLE,
    units=UNITS,
)

effective_specorder = OVERRIDE_SPECORDER or specorder_from_atoms(atoms)
print(f"Exported snapshot {SNAPSHOT_INDEX} from {VASPRUN_FILE} to {output_path}")
print(f"Number of atoms: {len(atoms)}")
print(f"Specorder: {effective_specorder}")
print(f"Velocities attached: {atoms.get_velocities() is not None}")
